# Grouped summaries and joins
Run from this folder, with data/ beside the notebook. Classroom examples and worked answers.


## Four households, two regions
Open 03_summaries.ipynb. All amounts are synthetic EUR per month.

In [ ]:
import pandas as pd
small = pd.DataFrame({
    "id": ["A", "B", "C", "D"],
    "code": ["N", "N", "S", "S"],
    "income": [1000, 3000, 2000, 4000],
    "saving": [-100, 600, 400, 600],
})
print(small.to_string(index=False))

## One column, three summaries
Mean saving is 375 EUR per household. Total saving is 1500 EUR.

In [ ]:
print("Mean:", small["saving"].mean())
print("Median:", small["saving"].median())
print("Total:", small["saving"].sum())

## Several operations with agg
The list selects operations. The columns select variables.

In [ ]:
totals = small[["income", "saving"]].agg(
    ["mean", "median", "sum"]
)
print(totals.to_string())

## A mean within each region
Each region becomes one row in the result.

In [ ]:
regional = small.groupby(
    "code", as_index=False
)["saving"].mean()
print(regional.to_string(index=False))

## The calculation behind groupby
North: (-100 + 600) / 2. South: (400 + 600) / 2.

In [ ]:
north = small.loc[small["code"] == "N", "saving"]
south = small.loc[small["code"] == "S", "saving"]
print("North:", north.tolist(), north.mean())
print("South:", south.tolist(), south.mean())

## Names for the output columns
Each pair gives the input column and the operation.

In [ ]:
regional = small.groupby("code", as_index=False).agg(
    households=("id", "size"),
    mean_saving=("saving", "mean"),
    total_saving=("saving", "sum"),
)
print(regional.to_string(index=False))

## The summary is another DataFrame
You can select, calculate, sort, or plot the result.

In [ ]:
print(regional.columns.tolist())
print(regional["mean_saving"].tolist())
print(regional["households"].sum())

## Sort the regional comparison
South appears first because its mean saving is larger.

In [ ]:
ranked = regional.sort_values(
    "mean_saving", ascending=False
)
print(ranked[["code", "mean_saving"]].to_string(
    index=False
))

## Rows and observed values
size counts rows. count counts non-missing values in its input column.

In [ ]:
observed = small.copy()
observed.loc[0, "saving"] = float("nan")
counts = observed.groupby("code", as_index=False).agg(
    rows=("id", "size"),
    saving_values=("saving", "count"),
)
print(counts.to_string(index=False))

## A comparison creates a Boolean column
True contributes 1 and False contributes 0 to the sum.

In [ ]:
small["shortfall"] = small["saving"] < 0
print(small[["id", "saving", "shortfall"]].to_string(
    index=False
))
print("Shortfalls:", small["shortfall"].sum())

## A Boolean mean is a proportion
One of two northern households has negative saving: 1 / 2 = 0.5.

In [ ]:
rates = small.groupby("code", as_index=False).agg(
    households=("id", "size"),
    shortfall_rate=("shortfall", "mean"),
)
print(rates.to_string(index=False))

## Exercise 1: a regional comparison
Work in the notebook. Predict the two median saving values before running.

In [ ]:
small["saving_rate"] = (
    small["saving"] / small["income"]
)
# Group by code.
# Report households, median saving, and mean saving_rate.
# Sort the result by median saving, largest first.

## Exercise 1: worked answer
The median combines the middle values within each region.

In [ ]:
answer = small.groupby("code", as_index=False).agg(
    households=("id", "size"),
    median_saving=("saving", "median"),
    mean_rate=("saving_rate", "mean"),
)
answer = answer.sort_values("median_saving", ascending=False)
print(answer.round(3).to_string(index=False))

## A small denominator example
The households save 20% and 10% of their incomes.

In [ ]:
two = pd.DataFrame({
    "income": [1000, 3000],
    "saving": [200, 300],
})
two["rate"] = two["saving"] / two["income"]
print(two.to_string(index=False))

## Mean rate and pooled rate
15% gives equal weight to households. 12.5% divides combined saving by combined income.

In [ ]:
mean_rate = two["rate"].mean()
pooled_rate = two["saving"].sum() / two["income"].sum()
print(f"Mean household rate: {mean_rate:.1%}")
print(f"Pooled saving rate:  {pooled_rate:.1%}")

## A separate table of region names
The shared code connects each household to one region name.

In [ ]:
names = pd.DataFrame({
    "code": ["N", "S"],
    "region": ["North", "South"],
})
print(names.to_string(index=False))
print("Unique codes:", names["code"].is_unique)

## A left join keeps the household rows
Many households match one row in the region table.

In [ ]:
joined = small.merge(
    names, on="code", how="left",
    validate="many_to_one",
)
print(joined[["id", "code", "region"]].to_string(
    index=False
))

## A duplicated lookup row multiplies data
Both northern households now match two lookup rows.

In [ ]:
bad_names = pd.concat(
    [names, names.iloc[[0]]], ignore_index=True
)
unchecked = small.merge(bad_names, on="code", how="left")
print("Households:", len(small))
print("Joined rows:", len(unchecked))

## Validation catches the duplicate key
The error stops us from using the six-row result.

In [ ]:
try:
    small.merge(
        bad_names, on="code", how="left",
        validate="many_to_one",
    )
except pd.errors.MergeError as error:
    print(type(error).__name__)

## An unmatched code remains visible
The left join retains household A. Its missing region exposes the unmatched code.

In [ ]:
unknown = small.copy()
unknown.loc[0, "code"] = "X"
joined = unknown.merge(names, on="code", how="left")
missing = joined.loc[joined["region"].isna(), ["id", "code"]]
print(missing.to_string(index=False))

## The full classroom sample
The helper repeats Lecture 8's preparation and adds the region names.

In [ ]:
from lesson_data import load_households
df = load_households()
print("Households:", len(df))
print("Defined saving shares:", df["saving_share"].count())
print(df[["region", "saving"]].head(3).to_string(index=False))

## A summary of 195 households
Each regional count contributes to the full sample of 195 households.

In [ ]:
summary = df.groupby("region", as_index=False).agg(
    households=("household_id", "size"),
    mean_saving=("saving", "mean"),
    median_saving=("saving", "median"),
)
print(summary.round(1).to_string(index=False))
assert summary["households"].sum() == len(df)

## Exercise 2: combine two summaries
Return four rows. Check that the household counts still sum to 195.

In [ ]:
df["shortfall"] = df["saving"] < 0
# Build one table: region, households, mean_saving.
# Build another: region, shortfall_rate.
# Join them on region, using validate="one_to_one".
# Sort by shortfall_rate, largest first.

## Exercise 2: worked answer
One-to-one checks that each region appears once in each summary.

In [ ]:
rates = df.groupby("region", as_index=False).agg(
    shortfall_rate=("shortfall", "mean"),
)
report = summary.merge(
    rates, on="region", how="left", validate="one_to_one"
).sort_values("shortfall_rate", ascending=False)
columns = ["region", "households", "mean_saving", "shortfall_rate"]
print(report[columns].round(3).to_string(index=False))
assert report["households"].sum() == len(df)